# BrickOrSwish: video → detections.json + annotated video

Runs on Google's machine, not yours. Steps: upload a video, click the rim 3 times, then COCO-pretrained
**RF-DETR Nano** (Apache-2.0) detects the ball on every frame. You get:

| File | What |
|---|---|
| `hoop.json` | your 3 clicks (`shared/schema/hoop.schema.json`) |
| `detections.json` | per-frame boxes (`shared/schema/detections.schema.json`); ball is `"ball"`, other COCO classes keep their name |
| `events.json` | shots found by our Python shot logic (`shared/schema/events.schema.json`) |
| `annotated.mp4` | boxes (orange = ball the logic uses, dark orange = ball it ignores), hoop ellipse, approach zone, tracked ball (green), state and make/miss |

**Security:** the setup installs pinned versions (`ml/requirements-colab.txt` once it exists), then the audit cell
checks PyTorch is new enough and checks every installed dependency against OSV.dev. It stops if anything is vulnerable.

**First:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all, or run the cells one by one.
The uploaded video lives only on this temporary machine and is gone when the runtime is disconnected.

In [ ]:
# Settings
BRANCH = "feat/shot-logic-reference"  # branch of github.com/Pietrzaaq/swish-or-brick to run
RFDETR_VERSION = "1.12.1"             # used only until ml/requirements-colab.txt (all versions pinned) exists
ALLOW_VULNS = False                   # True = continue even if the security audit finds known vulnerabilities
CROP = "hoop"       # "hoop": detector sees a square around the rim (the ball is bigger); "full": whole frame
THRESHOLD = 0.15    # keep low: the shot logic filters at ball_min_conf (0.3) itself
HOOP_FRAME = 0      # frame to click the rim on (pick one where the rim is clearly visible)
HOOP = None         # paste a hoop.json dict here to skip clicking, e.g. {"image_width": 1920, ...}

In [ ]:
# Setup: code + detector library (1-3 minutes)
import os, sys
!nvidia-smi -L || echo "NO GPU: Runtime > Change runtime type > T4 GPU (CPU works but is very slow)"
!rm -rf /content/swish-or-brick && git clone -q --depth 1 -b {BRANCH} https://github.com/Pietrzaaq/swish-or-brick.git /content/swish-or-brick
sys.path.insert(0, "/content/swish-or-brick/ml")
PINS = "/content/swish-or-brick/ml/requirements-colab.txt"
if os.path.exists(PINS):
    !pip install -q -r {PINS}
else:
    print("No pinned versions in the repo yet: installing rfdetr only. The audit cell writes the pin file.")
    !pip install -q rfdetr=={RFDETR_VERSION}

import subprocess
def run_py(*args):
    # python -m <args> in a fresh process, streaming its output. Colab preloads some libraries (e.g. an old
    # Pillow) into this notebook's process at startup; a fresh process loads only what is installed now.
    proc = subprocess.Popen([sys.executable, "-u", "-m", *map(str, args)], cwd="/content/swish-or-brick/ml",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

In [ ]:
# Security audit
# 1. PyTorch >= 2.10: older versions let a crafted .pth file run code even in safe mode (CVE-2026-24747).
# 2. Every installed package the detector depends on, at its exact version, checked against OSV.dev
#    (CVE / GitHub / PyPI advisories and known malicious packages). Vulnerable ones are upgraded to the
#    smallest fixed release and checked again. Writes /content/requirements-colab.txt.
# Runs in a fresh process (run_py), so it checks exactly what the detector will load.
from importlib.metadata import version
from brickorswish.tools.deps_audit import version_tuple

if version_tuple(version("torch")) < (2, 10):
    print(f"torch {version('torch')} is older than 2.10, upgrading")
    !pip install -q -U "torch>=2.10" torchvision
assert version_tuple(version("torch")) >= (2, 10), f"torch {version('torch')} is still older than 2.10"
print(f"torch {version('torch')} OK")

ROOTS = ["rfdetr", "opencv-python", "opencv-python-headless", "opencv-contrib-python"]  # missing ones are skipped
rc = run_py("brickorswish.tools.deps_audit", *ROOTS, "--pins", "/content/requirements-colab.txt", "--fix")
if rc != 0 and not ALLOW_VULNS:
    raise RuntimeError("Audit failed (see above). Stop here, or set ALLOW_VULNS = True if you accept the risk.")
print("audit clean")

In [ ]:
# Upload the video (or mount Drive and set VIDEO to its path instead; faster for big files)
from google.colab import files
uploaded = files.upload()
VIDEO = "/content/" + next(iter(uploaded))
print(VIDEO)

In [ ]:
# Hoop calibration: click LEFT rim edge, RIGHT rim edge, then the FRONT edge (nearest the camera)
import json
from brickorswish.detect.colab import pick_hoop
from brickorswish.detect.video import read_frame

frame = read_frame(VIDEO, HOOP_FRAME)
hoop = HOOP or pick_hoop(frame)
print(json.dumps(hoop))  # copy this into HOOP above to reuse it

In [ ]:
# Check: yellow ellipse should sit on the rim; grey square = what the detector sees
import cv2
from IPython.display import Image, display
from brickorswish.detect.annotate import draw_hoop_preview
from brickorswish.detect.crop import watch_crop
from brickorswish.logic import ShotLogicConfig
from brickorswish.paths import CONFIG_V1

cfg = ShotLogicConfig.load(CONFIG_V1)
preview = draw_hoop_preview(frame, hoop, cfg, watch_crop(hoop, cfg) if CROP == "hoop" else None)
scale = 900 / max(preview.shape[:2])
small = cv2.resize(preview, None, fx=scale, fy=scale) if scale < 1 else preview
display(Image(data=cv2.imencode(".jpg", small)[1].tobytes()))  # no Pillow in this process

In [ ]:
# Run: detection (a few minutes per minute of 1080p video on a T4), shot logic, annotated video
# Runs in a fresh process (run_py) with the audited library versions.
from pathlib import Path

OUT = Path("/content/out") / Path(VIDEO).stem
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "hoop.json").write_text(json.dumps(hoop))
if run_py("brickorswish.detect.run_video", VIDEO, "--hoop", OUT / "hoop.json", "--out", OUT,
          "--crop", CROP, "--threshold", THRESHOLD) != 0:
    raise RuntimeError("Run failed, see the error above.")

In [ ]:
# Watch the annotated video here (skipped above 50 MB; download it instead)
import base64
from IPython.display import HTML, display

mp4 = OUT / "annotated.mp4"
if mp4.stat().st_size < 50e6:
    b64 = base64.b64encode(mp4.read_bytes()).decode()
    display(HTML(f'<video controls style="max-width:100%" src="data:video/mp4;base64,{b64}"></video>'))
else:
    print(f"{mp4.stat().st_size / 1e6:.0f} MB, download it with the next cell")

In [ ]:
# Download everything as one zip (also contains requirements-colab.txt: commit it as ml/requirements-colab.txt
# so the next runs install exactly these versions)
import shutil
shutil.copy("/content/requirements-colab.txt", OUT / "requirements-colab.txt")
zip_path = shutil.make_archive(str(OUT), "zip", OUT)
files.download(zip_path)